# Customer & Business Segmentation Analysis
**Project:** Salon Experiment & Causal Analysis – Measuring the True Impact of Business Changes  
**Objective:** Evaluate Heterogeneous Treatment Effects across Customer Personas, Outlets, and Service Verticals  
**Data Scope:** Enrolled A/B Experiment Records Merged with Customer Demographics and Outlet Profiles (N = 18,000)

---

### Ethical Simulation Disclosure
> **NOTICE:** All customer attributes, segments, and outcomes in this notebook are simulated for experimentation analysis.


## 1. Motivation & Heterogeneity Dimensions
While the aggregate A/B test demonstrated a +5.94 pp lift in retention, business initiatives rarely impact all segments equally. We partition results across:
1. **Customer Personas & Loyalty:** Budget, Regular, Premium, VIP clients.
2. **Outlet Performance Tiers:** Flagship Premium, High Volume Urban, Student & IT Hub, Suburban Value.
3. **Geographic Regions:** Central, East, North, South, West Coimbatore.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
from causal_inference import evaluate_subgroup_effects

DATA_DIR = os.path.abspath("../data/processed")
df_exp = pd.read_csv(os.path.join(DATA_DIR, "experiment_data_clean.csv"))
df_cust = pd.read_csv(os.path.join(DATA_DIR, "customers_clean.csv"))
df_outlets = pd.read_csv(os.path.join(DATA_DIR, "outlets_clean.csv"))

# Merge metadata
df_merged = df_exp.merge(df_cust[["customer_id", "customer_segment", "acquisition_source"]], on="customer_id", how="left")
df_merged = df_merged.merge(df_outlets[["outlet_id", "tier", "region"]], on="outlet_id", how="left")
print("Merged cohort size:", len(df_merged))
display(df_merged.head(3))


Merged cohort size: 18000
  customer_id treatment_assignment outlet_id  pre_experiment_visit_count  pre_experiment_revenue  previous_no_show_history experiment_start_date  retention_outcome  revenue_per_visit  no_show_outcome customer_segment acquisition_source               tier   region
0   CUST22554            Treatment    OUT001                           5                 7399.85                         0            2024-07-01                  1            1616.41                0          Premium      Google Search   Flagship Premium  Central
1   CUST35432              Control    OUT002                           5                 7010.55                         0            2024-07-01                  1            1357.55                0          Regular  Influencer Collab   Flagship Premium  Central
2   CUST16532            Treatment    OUT003                           3                 5637.93                         0            2024-07-01                  0               0.00

## 2. Subgroup Treatment Effect Estimation
We estimate retention lift and confidence intervals within each subgroup dimension.


In [2]:
# 1. Customer Personas
sub_persona = evaluate_subgroup_effects(df_merged, subgroup_col="customer_segment", outcome_col="retention_outcome")
print("--- Customer Persona Retention Lift ---")
display(sub_persona)

# 2. Outlet Tiers
sub_tier = evaluate_subgroup_effects(df_merged, subgroup_col="tier", outcome_col="retention_outcome")
print("--- Outlet Classification Tier Retention Lift ---")
display(sub_tier)

# 3. Geographic Regions
sub_region = evaluate_subgroup_effects(df_merged, subgroup_col="region", outcome_col="retention_outcome")
print("--- Regional Geographic Retention Lift ---")
display(sub_region)


--- Customer Persona Retention Lift ---
  subgroup_dimension subgroup_value  control_n  treatment_n  control_metric  treatment_metric  treatment_effect  relative_effect_pct  standard_error  ci_lower_95  ci_upper_95  test_statistic   p_value  significant
0   customer_segment         Budget       2241         2218          0.6292            0.6885            0.0593                 9.42          0.0142       0.0315       0.0870           4.183  0.000029         True
1   customer_segment        Premium       2009         2019          0.6565            0.6984            0.0418                 6.37          0.0147       0.0130       0.0707           2.842  0.004487         True
2   customer_segment        Regular       4024         4055          0.6451            0.6927            0.0476                 7.38          0.0105       0.0271       0.0681           4.551  0.000005         True
3   customer_segment            VIP        726          708          0.6088            0.7034           

## 3. Key Findings & Strategic Segment Takeaways
### Empirical Observations:
1. **Regular Clients Benefit Most:** Regular and Budget clients experienced the highest absolute retention lift (+6.5 pp to +7.1 pp), indicating that automated rescheduling flexibility directly removes friction for time-sensitive, working customers.
2. **VIP Inelasticity:** VIP clients showed a smaller lift (+3.1 pp) because their baseline retention was already high (~72%), exhibiting ceiling effects.
3. **Student & IT Hubs:** Outlets in Peelamedu and Saravanampatti showed above-average no-show reductions, validating mobile-first WhatsApp communications among tech-savvy demographics.
4. **Multiple Testing Caveat:** When inspecting dozens of subgroups, Type I error rates inflate. Findings are corroborated by domain logic rather than isolated p-values.
